In [7]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os

In [8]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input thesis boundary polygon that shoreline feature class will be clipped to
boundary = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\MASTER_THESIS.gdb\boundary_thesis_p26920"

# Input feature class containing shoreline polygons for the entire Caribbean
raw_shore_fc = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Raw_data\Caribbean_Shorelines_raw.gdb\Caribbean_shorelines"

# Output renamed, reprojected, and clipped shorelines polygon feature classes
all_shore_p26920_fc = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\Caribbean_shorelines_all_p26920"
all_shore_g4326_fc = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\Caribbean_shorelines_all_g4326"
clip_shore_p26920_fc = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\CARIBBEAN_SHORELINES_THESIS_p26920"

# Output shapefiles of shoreline feature classes (shapefiles saved outside of geodatabase and can be used in analyses outside of ArcGIS)
all_shore_p26920_shp = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\exported_files\shapefiles\Caribbean_shorelines_all_shapefile_p26920\Caribbean_shorelines_all_shapefile_p26920.shp"
all_shore_g4326_shp = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\exported_files\shapefiles\Caribbean_shorelines_all_shapefile_g4326\Caribbean_shorelines_all_shapefile_g4326.shp"
clip_shore_p26920_shp = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\exported_files\shapefiles\CARIBBEAN_SHORELINES_THESIS_SHAPEFILE_p26920\CARIBBEAN_SHORELINES_THESIS_SHAPEFILE_p26920.shp"

In [9]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [10]:
# ── 1. EXPORT RENAMED RAW SHORELINES FEATURE CLASS TO SHORELINES GDB ─────────────────────────────────

# Export Features (Conversion Tools): Function
try:
    arcpy.conversion.ExportFeatures(
        in_features=raw_shore_fc,
        out_features=all_shore_p26920_fc
    )
    print(f"✅ Exported shapefile: {all_shore_p26920_fc} \n")

# print warning if tool failed
except Exception as e:
    print(f"❌[FAILURE] Feature class failed to export: {e} \n")

✅ Exported shapefile: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\Caribbean_shorelines_all_p26920 



In [11]:
# ── 2. REPROJECT RAW SHORELINES FEATURE CLASS TO EPSG:4326 SINCE SPANS ENTIRE CARIBBEAN ──────────────

# Project (Data Management Tools):
try:
    arcpy.management.Project(
        in_dataset=all_shore_p26920_fc,
        out_dataset=all_shore_g4326_fc,
        out_coor_system=arcpy.SpatialReference(4326),
        # ** transformations are BI-DIRECTIONAL, tool will apply correct conversion based on input and output coordinate systems **
        transform_method="WGS_1984_(ITRF00)_To_NAD_1983",
        in_coor_system=arcpy.SpatialReference(26920),
        # "PRESERVE_SHAPE" adds extra vertices to output polygons as needed to make reprojected shape more accurate
        preserve_shape="PRESERVE_SHAPE"
    )
    print(f"✅ Reprojected: {os.path.basename(all_shore_g4326_fc)}")

# print warning message if tool fails
except Exception as e:
    print(f"\n❌[FAILURE] Reprojection failed: {e} \n")

✅ Reprojected: Caribbean_shorelines_all_g4326


In [12]:
# ── 3. CLIP SHORELINES FEATURE CLASS TO THESIS BOUNDARY EXTENT ───────────────────────────────────────

# Set environment parameters for tool before running it
arcpy.env.maintainCurveSegments=True


# Pairwise Clip (Analysis Tools):
try:
    arcpy.analysis.PairwiseClip(
        in_features=all_shore_p26920_fc,
        clip_features=boundary,
        out_feature_class=clip_shore_p26920_fc
        # cluster_tolerance defaults to input feature class spatial reference x,y tolerance (tool documentation discourages editing this parameter)
        # precision defaults to precision of first input feature class (use "MAX_PRECISION" if want tool to apply higher level of accuracy than defult input spatial reference; all_shore_p26920_fc already has ~22nm res and 1mm tolerance)
    )
    print(f"✅ Created clipped shorelines feature class: {clip_shore_p26920_fc} \n")

# print warning if tool failed
except Exception as e:
    print(f"❌[FAILURE] Clipped shorelines feature class creation failed: {e} \n")


# Clear settings in environment for subsequent blocks
arcpy.ClearEnvironment("maintainCurveSegments")

✅ Created clipped shorelines feature class: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\CARIBBEAN_SHORELINES_THESIS_p26920 



In [13]:
##### NOTE: IF WANT TO ADD METADATA TO OUTPUT *CLIPPED* AND *REPROJECTED* FEATURE CLASSES BEFORE EXPORTING AS SHAPEFILES SO IT IS INCLUDED IN SHP.XML FILES, DO IT NOW (IN CATALOG VIEW) #####

In [14]:
# ── 4. EXPORT SHORELINES FEATURE CLASSES AS SHAPEFILES (TO USE IN MAXENT OUTSIDE ARCGIS) ─────────────

# Export Features (Conversion Tools): Function
def export_fc_to_shp(in_fc, out_shp):

    try:
        arcpy.conversion.ExportFeatures(
            in_features=in_fc,
            out_features=out_shp
        )
        print(f"✅ Exported shapefile: {out_shp} \n")

    # print warning if tool failed
    except Exception as e:
        print(f"❌[FAILURE] Shapefile failed to export for {os.path.basename(out_shp)}: {e} \n")


# Call function to export clipped shorelines feature class (*to use in RStudio for MaxEnt model) + export raw unclipped shorelines feature class in EPSG:26920 and EPSG:4326 as shapefiles
export_fc_to_shp(all_shore_p26920_fc, all_shore_p26920_shp)
export_fc_to_shp(all_shore_g4326_fc, all_shore_g4326_shp)
export_fc_to_shp(clip_shore_p26920_fc, clip_shore_p26920_shp)

✅ Exported shapefile: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\exported_files\shapefiles\Caribbean_shorelines_all_shapefile_p26920\Caribbean_shorelines_all_shapefile_p26920.shp 

✅ Exported shapefile: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\exported_files\shapefiles\Caribbean_shorelines_all_shapefile_g4326\Caribbean_shorelines_all_shapefile_g4326.shp 

✅ Exported shapefile: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\exported_files\shapefiles\CARIBBEAN_SHORELINES_THESIS_SHAPEFILE_p26920\CARIBBEAN_SHORELINES_THESIS_SHAPEFILE_p26920.shp 

